# rl-trading-live — Colab Master Notebook
Run cells 1-7 in order to train. Cell 8 = dashboard, 9 = crash recovery, 10 = GPU profiling.
Code lives in GitHub; data + checkpoints live in Google Drive (Colab is ephemeral).

**Fresh "Run all" works top-to-bottom with no manual edits.** The repo is CLONED/UPDATED
(cell 3) BEFORE dependencies are installed (cell 4), because install reads the repo's
requirements.txt — installing first would fail on a fresh runtime where the repo dir does
not yet exist.

## COLAB RUN ORDER — READ FIRST (humans & LLMs)

Run the cells **in order, top to bottom**. Code lives in GitHub; data +
checkpoints live in Google Drive (Colab storage is ephemeral). Each step below
lists **what it does**, the **success signal** to look for, and the **most
common failure + its one-line fix**.

> ⚠️ **If you RESTART the runtime or it TIMES OUT, Google Drive UNMOUNTS.**
> You **MUST re-run Cell 2 (MOUNT DRIVE)** before Cell 6, or training dies with
> `FileNotFoundError` on the CSV. That error is an unmounted Drive — **not** a
> code bug. Do not start editing the data loader.

| # | Cell | What it does | Success signal | Most common failure → fix |
|---|------|--------------|----------------|---------------------------|
| 1 | **GPU CHECK** | Asserts an A100 GPU (>30 GB VRAM) is attached. | `GPU: A100 ... VRAM: 40.0GB` | Not an A100 → Runtime → Change runtime type → **A100 GPU**. |
| 2 | **MOUNT DRIVE** | Mounts Drive and confirms the CSV is visible (auto force-remounts once if stale). | `Drive mounted. Primary data file confirmed:` | File not found → open **RL-Trading-Data** in Drive, confirm the CSV name; re-run. If still stale: `drive.mount('/content/drive', force_remount=True)`. |
| 3 | **CLONE / UPDATE REPO** | Clones repo (fresh) or hard-resets to `origin/master`. **Runs BEFORE install.** | `Repo cloned.` / `Repo hard-reset to origin/master.` | Network blip → re-run the cell. |
| 4 | **INSTALL DEPS** | `pip install -r requirements.txt` (reads the cloned repo). | `Dependencies installed. TA-Lib ... import OK.` | `talib` ModuleNotFound → re-run Cell 3 then Cell 4 (install needs the repo dir). |
| 4b | **SANITY IMPORT** | Imports core modules so a broken dep fails loudly here. | `Core modules import OK — ready to train.` | Import error → re-run Cell 3+4. |
| 5 | **CLEAN MANIFEST** | Removes stale checkpoint entries from `gpu/manifest.json`. | `Manifest cleaned: N → M entries.` | No manifest yet → harmless, created on first train. |
| 6 | **SYSTEM INSPECTION** | Runs `inspect_system.py` preflight; aborts on ❌. | All checks ✅/⚠️, exit 0. | Any ❌ → read its IRAC block, fix, re-run. |
| 7 | **RUN TRAINING** | Resumes from Drive checkpoints and trains. | `DAY n 🟢 ... Episode n ...` lines stream. | `FileNotFoundError` CSV → **re-run Cell 2** (Drive unmounted). No checkpoint → fresh start is normal. |
| 8 | (opt) Dashboard | Streamlit UI via localtunnel/ngrok. | URL prints. | Tunnel flaky → use the ngrok fallback line. |
| 9 | (opt) Crash recovery | Finds the best valid checkpoint after a crash. | Best checkpoint printed. | — then re-run Cell 7. |
| 10 | (opt) GPU profiling | Profiles a forward pass. | `GPU utilization OK`. | <50% → raise `BATCH_SIZE_ENV`. |

**Order rules that bite people:**
- **Clone (Cell 3) BEFORE install (Cell 4)** — install reads the repo's `requirements.txt`.
- The **`--manifest` must live in the `gpu/` dir** (next to the checkpoints) so resume can find them.
- After **any** runtime restart/timeout: **Cell 2 again** before Cell 6.

Full troubleshooting table: **`docs/COLAB_RUNBOOK.md`** in the repo.


In [ ]:
# CELL 1 — GPU CHECK
import torch
assert torch.cuda.is_available(), 'NO GPU DETECTED — switch to A100 runtime'
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {gpu_name} | VRAM: {vram_gb:.1f}GB')
assert vram_gb > 30, f'Expected A100 (>30GB), got {vram_gb:.1f}GB — check runtime'

In [ ]:
# CELL 2 — MOUNT DRIVE (idempotent + self-healing; safe to re-run)
# WHY THIS IS HARDENED: a fresh OR restarted/timed-out Colab runtime UNMOUNTS
# Google Drive, so on a re-run the data path silently resolves to nothing and
# training later dies with a confusing FileNotFoundError on the CSV. To stop
# that: mount, then VERIFY the data file is visible; if it is not, retry ONCE
# with force_remount=True (a stale half-mount is NOT refreshed by the default
# force_remount=False — that is exactly the trap that caused the crash), then
# assert with a clear message naming the exact path so the user knows precisely
# what to check in Drive.
from google.colab import drive
import os

DATA_FILE = ('/content/drive/MyDrive/RL-Trading-Data/'
             'EURUSD_M1_202101131130_202605270000_2020_2026.csv')

drive.mount('/content/drive')  # default: reuses an existing mount if present

if not os.path.exists(DATA_FILE):
    # Most common cause: a stale mount from a previous (now-restarted) session.
    # force_remount=True tears it down and re-establishes it cleanly.
    print('Data file not visible after mount — forcing a clean remount...')
    drive.mount('/content/drive', force_remount=True)

assert os.path.exists(DATA_FILE), (
    'PRIMARY DATA FILE NOT FOUND after mounting Drive.\n'
    f'  Expected: {DATA_FILE}\n'
    '  Open the RL-Trading-Data folder in Drive and confirm the CSV is present '
    'and named EXACTLY as above. Then re-run this cell.'
)
print(f'Drive mounted. Primary data file confirmed:\n  {DATA_FILE}')


In [ ]:
# CELL 3 — CLONE OR UPDATE REPO (always gets latest code from GitHub)
# MUST run BEFORE the install cell: install reads this repo's requirements.txt,
# so the repo directory has to exist first. On a fresh "Run all" the dir is absent
# and we clone; on a re-run it exists and we hard-reset to origin/master (which
# discards any local patches a tool like Gemini may have applied in Colab).
import os, subprocess, sys

REPO_URL  = 'https://github.com/monty313/rl-trading-live.git'
CLONE_DIR = '/content/rl-trading-live'

if os.path.exists(CLONE_DIR):
    # Hard reset — discard any local patches Gemini may have applied
    subprocess.run(['git', '-C', CLONE_DIR, 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', CLONE_DIR, 'reset', '--hard', 'origin/master'], check=True)
    print('Repo hard-reset to origin/master.')
else:
    subprocess.run(['git', 'clone', REPO_URL, CLONE_DIR], check=True)
    print('Repo cloned.')

# Always cd to repo root so relative imports and ! commands work
os.chdir(CLONE_DIR)
sys.path.insert(0, CLONE_DIR)

# Flush stale module cache so updated code is actually imported
for m in list(sys.modules):
    if m.startswith(('core', 'training', 'backtest', 'broker', 'jordan')):
        del sys.modules[m]

# Confirm we have the latest commit
r = subprocess.run(['git', 'log', '--oneline', '-3'], capture_output=True, text=True, cwd=CLONE_DIR)
print('Latest commits:\n' + r.stdout)

In [ ]:
# CELL 4 — INSTALL DEPENDENCIES (once per session; runs AFTER the repo is cloned)
# Deps are pinned in /content/rl-trading-live/requirements.txt, which now installs
# cleanly on a STANDARD Colab runtime with NO system C libraries to build:
#   • TA-Lib>=0.6.7 ships PREBUILT manylinux cp311/cp312 wheels, so `pip install`
#     needs no apt package and no TA-Lib C source build (0.4.x did — that is what
#     broke the old install). The apt-get below is therefore just a harmless no-op
#     fallback (check=False) for exotic platforms; on Colab it does nothing useful.
#   • numpy is pinned <3.0 (NOT <2.0) so it COEXISTS with Colab's preinstalled
#     numpy 2.x instead of forcing a downgrade that conflicted with the CUDA stack.
#   • faiss-cpu>=1.9 and torch>=2.2 (floor only) keep Colab's own CUDA torch wheel.
# NOTE: we deliberately do NOT pass `-q` so real pip errors surface, and we keep
# check=True so any genuine failure stops the run loudly.
import subprocess, sys
subprocess.run(['apt-get', 'install', '-y', '-q', 'ta-lib'], check=False)  # no-op fallback
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r',
                '/content/rl-trading-live/requirements.txt'], check=True)

# Verify TA-Lib actually imported (the historical failure point).
try:
    import talib
    print(f'Dependencies installed. TA-Lib {talib.__version__} import OK.')
except Exception as e:
    raise ImportError(
        'TA-Lib import FAILED after install — the indicator stack cannot run. '
        f'Underlying error: {e!r}'
    )

In [ ]:
# CELL 4b — SANITY IMPORT (fail fast if any core module is still missing)
# Imports the modules training needs so a missing/broken dep surfaces here with a
# clear message rather than deep inside the training loop.
import importlib
for _mod in ('core.settings', 'core.pipeline', 'training.train'):
    try:
        importlib.import_module(_mod)
    except Exception as e:
        raise ImportError(f'Core module {_mod!r} failed to import: {e!r}')
print('Core modules import OK — ready to train.')

In [ ]:
# CELL 5 — CLEAN MANIFEST (removes stale DQN entries that no longer exist on Drive)
import json, os

MANIFEST = '/content/drive/MyDrive/RL-Trading-Checkpoints/gpu/manifest.json'
CKPT_DIR = '/content/drive/MyDrive/RL-Trading-Checkpoints/gpu'

if os.path.exists(MANIFEST):
    with open(MANIFEST) as f:
        manifest = json.load(f)

    before = len(manifest.get('checkpoints', {}))
    cleaned = {
        name: meta
        for name, meta in manifest.get('checkpoints', {}).items()
        if os.path.exists(os.path.join(CKPT_DIR, name))   # file must exist
        and meta.get('phase', 'unknown') != 'unknown'      # skip legacy DQN
    }
    manifest['checkpoints'] = cleaned
    after = len(cleaned)

    with open(MANIFEST, 'w') as f:
        json.dump(manifest, f, indent=2)
    print(f'Manifest cleaned: {before} entries → {after} entries.')
    for name, meta in cleaned.items():
        print(f"  {name}  phase={meta['phase']}  ep={meta['episode']}  phi={meta['phi']:.4f}")
else:
    print('No manifest found — will be created fresh when training starts.')

In [ ]:
# CELL 6 — SYSTEM INSPECTION (aborts on failure)
# Output streams live — smoke_train takes ~5-10 min on first torch.compile warmup.
# You will see each check print as it completes. Do NOT interrupt unless you see ❌.
%cd /content/rl-trading-live
import subprocess, sys
r = subprocess.run([sys.executable, 'inspect_system.py'], cwd='/content/rl-trading-live')
if r.returncode != 0:
    raise RuntimeError('inspect_system.py failed — fix issues above before training')

In [ ]:
# CELL 7 — ⚙️ PARAMETER DASHBOARD (dark theme, pure ipywidgets)
# ─────────────────────────────────────────────────────────────────────────────
# RUN-ALL SAFE. This cell builds the single `PARAMS` dict that the RUN TRAINING
# cell consumes. Edit knobs in the UI and click "Apply & Lock Parameters" — OR
# do nothing: on "Run all" the cell auto-applies the DEFAULTS and prints a notice,
# so training still launches with the trained settings.py defaults.
#
# All knobs + their defaults/min/max/options come from core/interpret/
# dashboard_utils.py, which mirrors core/settings.py + core/reward/shaper.py
# (SOURCE OF TRUTH = THE CODE). Reward knobs are nested under PARAMS["REWARD"].
# The right-hand pane shows the LIVE PARAMS JSON; section headers show a 🟢 dot
# when every knob in the group is at its default and 🟡 when something changed.
import json

from core.interpret.dashboard_utils import (
    SECTION_GROUPS, widget_specs, default_params, build_params,
    diff_from_defaults, params_hash,
)

try:
    import ipywidgets as W
    from IPython.display import display, HTML
    _HAVE_WIDGETS = True
except Exception:                       # headless / no-ipywidgets -> auto defaults
    _HAVE_WIDGETS = False

_SPECS = widget_specs()
_DEFAULTS = default_params()

# Dark theme palette (panel bg, header bg, accent per section).
_BG       = "#1e1e2e"
_PANEL_BG = "#272735"
_TEXT     = "#e6e6f0"
_ACCENT   = {"ftmo": "#f9a825", "reward": "#43a047", "streak": "#fb8c00",
             "risk": "#e53935", "training": "#3949ab", "gpu": "#00acc1"}


def _make_widget(key, spec):
    """Build ONE ipywidget for a spec entry, styled for the dark theme."""
    kind = spec["kind"]
    label = spec.get("label", key)
    style = {"description_width": "initial"}
    layout = W.Layout(width="98%")
    if kind in ("float", "floatlog"):
        step = float(spec.get("step", 0.01))
        return W.BoundedFloatText(value=float(spec["default"]),
                                  min=float(spec["min"]), max=float(spec["max"]),
                                  step=step, description=label, style=style,
                                  layout=layout)
    if kind == "int":
        return W.BoundedIntText(value=int(spec["default"]), min=int(spec["min"]),
                                max=int(spec["max"]), step=int(spec.get("step", 1)),
                                description=label, style=style, layout=layout)
    if kind == "dropdown":
        return W.Dropdown(options=spec["options"], value=spec["default"],
                          description=label, style=style, layout=layout)
    if kind == "checkbox":
        return W.Checkbox(value=bool(spec["default"]), description=label,
                          style=style, layout=layout, indent=False)
    return W.Text(value=str(spec["default"]), description=label, style=style,
                  layout=layout)


# ── _WIDGET_MAP: flat settings key -> its widget (the dashboard's backing store).
_WIDGET_MAP = {}
PARAMS = build_params(_DEFAULTS)        # always defined, even headless (Run-All safe)


def _build_params():
    """Assemble PARAMS from the CURRENT widget values (reward keys nested under
    PARAMS['REWARD']). Mirrors dashboard_utils.build_params on live widget state."""
    values = {k: w.value for k, w in _WIDGET_MAP.items()}
    return build_params(values)


if not _HAVE_WIDGETS:
    # Headless fallback: no UI possible. Use defaults, print the notice the spec
    # requires, and expose PARAMS so the train cell still runs end-to-end.
    PARAMS = build_params(_DEFAULTS)
    print("ℹ️  ipywidgets unavailable — using DEFAULT parameters (Run-All safe).")
    print("✅ Parameters locked. PARAMS ready for training.")
else:
    # Build every widget and group it under its section panel.
    for key, spec in _SPECS.items():
        _WIDGET_MAP[key] = _make_widget(key, spec)

    _preview = W.HTML(layout=W.Layout(width="100%"))
    _headers = {}     # group -> header HTML widget (for the 🟢/🟡 dot)

    def _group_changed(group):
        """True if ANY widget in this group differs from its default."""
        for key, spec in _SPECS.items():
            if spec.get("group") != group:
                continue
            if _WIDGET_MAP[key].value != _DEFAULTS[key]:
                return True
        return False

    def _refresh(_=None):
        """Recompute PARAMS, redraw the live JSON preview + each header dot."""
        global PARAMS
        PARAMS = _build_params()
        diff = diff_from_defaults(PARAMS, _DEFAULTS)
        h = params_hash(PARAMS)
        body = json.dumps(PARAMS, indent=2, default=str)
        _preview.value = (
            f"<div style='background:{_BG};color:{_TEXT};padding:10px;"
            f"border-radius:8px;font-family:monospace;font-size:11px;'>"
            f"<b>LIVE PARAMS</b> &nbsp; hash <code>{h}</code> &nbsp; "
            f"changed: <b>{len(diff)}</b>"
            f"<pre style='white-space:pre-wrap;max-height:560px;overflow:auto;'>"
            f"{body}</pre></div>")
        for group, title in SECTION_GROUPS:
            dot = "🟡" if _group_changed(group) else "🟢"
            acc = _ACCENT.get(group, "#666")
            _headers[group].value = (
                f"<div style='background:{acc};color:#000;padding:6px 10px;"
                f"border-radius:6px 6px 0 0;font-weight:700;'>{dot} {title}</div>")

    # Wire .observe on every widget so the preview + dots update live.
    for w in _WIDGET_MAP.values():
        w.observe(_refresh, names="value")

    # Assemble the six panels (left column ~65%).
    panels = []
    for group, title in SECTION_GROUPS:
        hdr = W.HTML()
        _headers[group] = hdr
        rows = [_WIDGET_MAP[k] for k, s in _SPECS.items() if s.get("group") == group]
        box = W.VBox([hdr] + rows,
                     layout=W.Layout(border=f"1px solid {_ACCENT.get(group,'#444')}",
                                     margin="0 0 10px 0", padding="0 0 8px 0",
                                     background_color=_PANEL_BG))
        panels.append(box)
    left = W.VBox(panels, layout=W.Layout(width="64%"))
    right = W.VBox([_preview], layout=W.Layout(width="35%", margin="0 0 0 1%"))

    _apply_btn = W.Button(description="Apply & Lock Parameters",
                          button_style="success",
                          layout=W.Layout(width="260px", height="40px"))
    _apply_out = W.Output()

    def _on_apply(_):
        global PARAMS
        PARAMS = _build_params()
        with _apply_out:
            _apply_out.clear_output()
            print("✅ Parameters locked. PARAMS ready for training.")

    _apply_btn.on_click(_on_apply)

    display(HTML(f"<style>.widget-label{{color:{_TEXT} !important;}}</style>"))
    display(W.HBox([left, right]))
    display(_apply_btn, _apply_out)
    _refresh()
    # Run-All safe: auto-lock defaults immediately so PARAMS is valid even if the
    # user never clicks the button (they can re-click to apply their edits).
    print("ℹ️  Auto-applied current values for Run-All. Click "
          "'Apply & Lock Parameters' after editing to re-lock.")
    print("✅ Parameters locked. PARAMS ready for training.")


In [ ]:
# CELL 7b — RUN TRAINING (consumes PARAMS from the dashboard cell)
# ─────────────────────────────────────────────────────────────────────────────
# RUN-ALL SAFE. The dashboard cell (CELL 7) defines `PARAMS`; this cell maps the
# subset of PARAMS that training/train.py accepts onto CLI flags (via
# dashboard_utils.params_to_cli) and adds the PATH/RESUME flags that are NOT
# dashboard knobs. Only flags whose value differs from the settings.py default are
# passed, so an untouched dashboard reproduces the trained-defaults launch.
#
# NOTE: torch.compile warmup makes the first ~10-15 min slow. NORMAL, not a crash.
import os, sys, subprocess

from core.interpret.dashboard_utils import params_to_cli

CLONE_DIR = '/content/rl-trading-live'
DRIVE     = '/content/drive/MyDrive'

# PARAMS comes from the dashboard cell; fall back to locked defaults if this cell
# is somehow run before it (keeps Run-All robust to cell-order edits).
try:
    PARAMS
except NameError:
    from core.interpret.dashboard_utils import default_params, build_params
    PARAMS = build_params(default_params())

# ── PATHS + RESUME (not dashboard knobs; edit here) ──────────────────────────
PATHS = {
    'csv':            f'{DRIVE}/RL-Trading-Data/EURUSD_M1_202101131130_202605270000_2020_2026.csv',
    'checkpoint-dir': f'{DRIVE}/RL-Trading-Checkpoints/gpu',
    'metrics-dir':    f'{DRIVE}/RL-Trading-Checkpoints/metrics',
    'manifest':       f'{DRIVE}/RL-Trading-Checkpoints/gpu/manifest.json',
    'snapshot-dir':   f'{DRIVE}/snapshots/params',   # PART 1 results-writer match dir
}
RESUME = {
    'resume':      True,    # resume WEIGHTS from the best Drive checkpoint
    'force-fresh': False,   # True = ignore checkpoints and start clean
}

# Dashboard knobs -> CLI flags (None when at the settings.py default, so omitted).
cli = params_to_cli(PARAMS)

_STORE_TRUE = {'resume', 'force-fresh', 'randomize-ftmo', 'randomize-ftmo-account'}
argv = [sys.executable, '-m', 'training.train']

# Valued path flags (always passed).
for flag, val in PATHS.items():
    if val is not None:
        argv += [f'--{flag}', str(val)]
# store_true resume flags.
for flag, val in RESUME.items():
    if val:
        argv.append(f'--{flag}')
# Dashboard-derived flags.
for flag, val in cli.items():
    if flag in _STORE_TRUE:
        if val:
            argv.append(f'--{flag}')
    elif val is not None:
        argv += [f'--{flag}', str(val)]

print('Launching:\n  ' + ' '.join(argv) + '\n', flush=True)
os.chdir(CLONE_DIR)
# check=False so a crash surfaces train.py's own diagnostic block, not a raw error.
subprocess.run(argv, cwd=CLONE_DIR, check=False)


In [ ]:
# CELL 7c — 📂 LOAD FROM CHECKPOINT (re-hydrate the dashboard from a saved cfg)
# ─────────────────────────────────────────────────────────────────────────────
# SELF-CONTAINED + RUN-ALL SAFE. Two ways to load a checkpoint's training cfg back
# onto the dashboard widgets:
#   MODE A  Drive path  — type a .pt path, click Load (torch.load map_location cpu)
#   MODE B  File upload — fallback FileUpload widget (reads bytes via io.BytesIO)
# Either way we read cfg from the checkpoint's "cfg" (or "config") key, coerce +
# clamp each value to the matching widget's type/range, write it onto the widget,
# show an HTML diff table (old -> new, ⚠️ on clamp, ✖ on skip), and rebuild PARAMS.
# Bulletproof: every failure prints an inline message; the cell NEVER throws.
import io

import torch
import ipywidgets as W
from IPython.display import display, HTML


def _coerce_for_widget(widget, raw):
    """Coerce+clamp `raw` to `widget`'s type/range. Returns (value, clamped, ok)."""
    try:
        if isinstance(widget, W.Checkbox):
            return (bool(raw), False, True)
        if isinstance(widget, W.Dropdown):
            for o in widget.options:
                try:
                    if raw == o or type(o)(raw) == o:
                        return (o, False, True)
                except (TypeError, ValueError):
                    continue
            return (None, False, False)                 # not an allowed option
        if isinstance(widget, W.BoundedIntText):
            v = int(round(float(raw)))
            cl = min(max(v, widget.min), widget.max)
            return (cl, cl != v, True)
        if isinstance(widget, W.BoundedFloatText):
            v = float(raw)
            cl = min(max(v, widget.min), widget.max)
            return (cl, cl != v, True)
    except (TypeError, ValueError):
        return (None, False, False)
    return (None, False, False)


def _apply_checkpoint_cfg(cfg_dict):
    """Apply a checkpoint cfg dict onto the dashboard widgets. Unpacks a nested
    'REWARD' sub-dict onto the flat widget keys, coerces/clamps each value, writes
    it onto the widget, and renders an HTML diff table. Rebuilds PARAMS at the end.
    Returns (applied, clamped, skipped) for inspection/tests."""
    if not isinstance(cfg_dict, dict):
        display(HTML("<b style='color:#e53935'>✖ checkpoint had no usable cfg "
                     "dict.</b>"))
        return ({}, [], [("<cfg>", "not a dict")])
    # Flatten REWARD onto the top level (matches how widgets are keyed).
    flat = {}
    for k, v in cfg_dict.items():
        if k == "REWARD" and isinstance(v, dict):
            flat.update(v)
        else:
            flat[k] = v
    applied, clamped, skipped = {}, [], []
    rows = []
    for key, raw in flat.items():
        widget = _WIDGET_MAP.get(key)
        if widget is None:
            skipped.append((key, "no matching widget"))
            continue
        old = widget.value
        val, was_clamped, ok = _coerce_for_widget(widget, raw)
        if not ok:
            skipped.append((key, "not coercible / not an allowed option"))
            continue
        widget.value = val
        applied[key] = val
        if was_clamped:
            clamped.append(key)
        if old != val:                      # only CHANGED rows go in the table
            status = "⚠️ clamped" if was_clamped else "changed"
            rows.append((key, old, val, status))
    matched = len(applied) - len(rows)      # applied but unchanged == matched default
    # Render the diff table: only changed rows (Parameter | Was | Now | Status).
    trs = "".join(
        f"<tr><td style='padding:2px 8px'>{k}</td>"
        f"<td style='padding:2px 8px;color:#888'>{o}</td>"
        f"<td style='padding:2px 8px;color:#43a047'>{n}</td>"
        f"<td style='padding:2px 8px'>{st}</td></tr>"
        for k, o, n, st in rows)
    skip_html = ""
    if skipped:
        skip_html = ("<div style='color:#fb8c00;margin-top:6px'>skipped: "
                     + ", ".join(f"{k} ({why})" for k, why in skipped) + "</div>")
    display(HTML(
        "<div style='background:#272735;color:#e6e6f0;padding:10px;"
        "border-radius:8px;font-family:monospace;font-size:12px'>"
        f"<b>Loaded {len(applied)} params, {len(rows)} changed, "
        f"{matched} matched defaults</b>"
        "<table style='border-collapse:collapse;margin-top:6px'>"
        "<tr><th style='text-align:left;padding:2px 8px'>Parameter</th>"
        "<th style='text-align:left;padding:2px 8px'>Was</th>"
        "<th style='text-align:left;padding:2px 8px'>Now</th>"
        "<th style='text-align:left;padding:2px 8px'>Status</th></tr>"
        f"{trs}</table>{skip_html}</div>"))
    # Rebuild PARAMS from the (now updated) widgets.
    try:
        global PARAMS
        PARAMS = _build_params()
        print("✅ Dashboard updated from checkpoint. PARAMS rebuilt.")
    except Exception as exc:
        print(f"⚠️  applied to widgets but could not rebuild PARAMS: {exc}")
    return (applied, clamped, skipped)


def _extract_cfg(ckpt):
    """Pull the cfg dict out of a loaded checkpoint object ('cfg' or 'config')."""
    if not isinstance(ckpt, dict):
        return None
    for key in ("cfg", "config"):
        if isinstance(ckpt.get(key), dict):
            return ckpt[key]
    return None


def _print_stats(ckpt, source):
    """Print the load path + a stats line from the checkpoint top-level keys
    (episodes / best_phi / pass_rate), or the spec's fallback when absent."""
    print(f"✅ Checkpoint loaded from: {source}")
    if isinstance(ckpt, dict) and any(
            k in ckpt for k in ("episode", "episodes", "global_ep",
                                 "best_phi", "phi", "pass_rate")):
        ep = ckpt.get("episodes", ckpt.get("episode", ckpt.get("global_ep", "?")))
        phi = ckpt.get("best_phi", ckpt.get("phi", "?"))
        pr = ckpt.get("pass_rate", "?")
        print(f"   episodes={ep}  best_phi={phi}  pass_rate={pr}")
    else:
        print("   (training stats not available in this checkpoint)")


def _handle_ckpt(ckpt, source):
    """Shared post-load: extract cfg (red/yellow inline errors), apply, print."""
    cfg = _extract_cfg(ckpt)
    if cfg is None:
        keys = list(ckpt.keys()) if isinstance(ckpt, dict) else []
        display(HTML(f"<b style='color:#f9a825'>⚠️ No cfg found in checkpoint. "
                     f"Keys present: {keys}</b>"))
        return
    _print_stats(ckpt, source)
    _apply_checkpoint_cfg(cfg)


def _load_from_path(path):
    import os
    if not os.path.exists(path):
        display(HTML(f"<b style='color:#e53935'>❌ File not found: {path}</b>"))
        return
    try:
        ckpt = torch.load(path, map_location="cpu", weights_only=False)
    except Exception:
        display(HTML("<b style='color:#e53935'>❌ Could not read checkpoint — "
                     "is this a valid .pt file?</b>"))
        return
    try:
        _handle_ckpt(ckpt, path)
    except Exception:                       # catch-all: full traceback, never crash
        import traceback
        print(traceback.format_exc())


def _load_from_bytes(name, data):
    try:
        ckpt = torch.load(io.BytesIO(data), map_location="cpu", weights_only=False)
    except Exception:
        display(HTML("<b style='color:#e53935'>❌ Could not read checkpoint — "
                     "is this a valid .pt file?</b>"))
        return
    try:
        _handle_ckpt(ckpt, f"upload:{name}")
    except Exception:
        import traceback
        print(traceback.format_exc())


# Guard: this cell needs the dashboard cell's _WIDGET_MAP/_build_params in scope.
if "_WIDGET_MAP" not in dir() or "_build_params" not in dir():
    print("ℹ️  Run the PARAMETER DASHBOARD cell (CELL 7) first, then this cell.")
else:
    _path_box = W.Text(placeholder="/content/drive/MyDrive/.../best_eval.pt",
                       description="Checkpoint path:",
                       style={"description_width": "initial"},
                       layout=W.Layout(width="70%"))
    _load_btn = W.Button(description="Load (Drive path)", button_style="info")
    _uploader = W.FileUpload(accept=".pt", multiple=False)
    _out = W.Output()

    def _on_load(_):
        with _out:
            _out.clear_output()
            p = (_path_box.value or "").strip()
            if not p:
                print("Enter a checkpoint path, or use the upload button below.")
                return
            _load_from_path(p)

    def _on_upload(change):
        with _out:
            _out.clear_output()
            up = change["new"]
            if not up:
                return
            # ipywidgets v8 -> tuple of dicts; v7 -> dict keyed by filename.
            item = up[0] if isinstance(up, (list, tuple)) else list(up.values())[0]
            name = item.get("name", "upload.pt")
            data = item.get("content")
            data = data.tobytes() if hasattr(data, "tobytes") else bytes(data)
            _load_from_bytes(name, data)

    _load_btn.on_click(_on_load)
    _uploader.observe(_on_upload, names="value")

    display(HTML("<b style='color:#e6e6f0'>📂 Load from Checkpoint</b>"))
    display(W.HBox([_path_box, _load_btn]))
    display(HTML("<span style='color:#888'>…or upload a .pt file:</span>"))
    display(_uploader, _out)


In [ ]:
# CELL 7d — 💾 SAVE SNAPSHOT (persist the locked PARAMS for the Compare panel)
# ─────────────────────────────────────────────────────────────────────────────
# SELF-CONTAINED + RUN-ALL SAFE. Saves the current PARAMS to a timestamped JSON
# snapshot the training results-writer (PART 1) later matches by params_hash, so
# each saved config can show how its run actually did. We ALSO append a one-line
# entry to a master snapshot_log.json (append, NEVER overwrite; duplicate_of when
# an identical-hash snapshot already exists). "View Log" renders the log as a
# table. This cell imports ONLY the stdlib + ipywidgets (no repo imports), and
# carries its OWN hardcoded _DEFAULTS so its diff is independent of other cells.
import json
import os
import hashlib
from datetime import datetime, timezone

import ipywidgets as W
from IPython.display import display, HTML

# Where snapshots live (must match the train cell's --snapshot-dir / CFG).
_SNAPSHOT_DIR = "/content/drive/MyDrive/snapshots/params"
_SNAPSHOT_LOG = os.path.join(_SNAPSHOT_DIR, "snapshot_log.json")

# OWN hardcoded defaults (flat key -> default), mirrored from settings.py at HEAD.
# Independent copy so this cell's diff_from_defaults stands alone (spec Task C).
_DEFAULTS = {
    "DAILY_TARGET_PCT": 0.025, "DAILY_MAX_DD_PCT": 0.010, "ACCOUNT_SIZE": 10000.0,
    "RANDOMIZE_FTMO_INPUTS": False, "BEAST_MODE": False,
    "pass_day_bonus": 2.0, "fail_day_penalty": -2.0, "ok_partial_lo": 0.25,
    "ok_partial_hi": 0.95, "exceed_scale": 1.0, "survival_bonus": 1.5,
    "red_day_scale": 1.0, "dd_efficiency_weight": 0.5,
    "streak_curve_a": 0.616998, "streak_curve_b": 0.221749, "streak_base": 0.5,
    "negative_streak_mult": 1.5, "mulligan_count": 1, "recovery_bonus": 3.0,
    "momentum_bonus": 0.2, "PASS_NO_BREACH_BONUS": 0.01, "PHASE_ADVANCE_STREAK": 10,
    "MAX_LOT": 2.0, "BARS_PER_DAY": 1440, "EPISODE_BARS": 43200, "LOOKBACK": 20,
    "SPEED_BONUS_MINUTES": 3, "speed_bonus": 0.3, "intraday_progress_scale": 0.5,
    "cross_day_giveback_scale": 0.5,
    "LR": 3e-4, "BATCH_SIZE_ENV": 64, "ROLLOUT_STEPS": 2048, "PPO_EPOCHS": 4,
    "GAMMA": 0.95, "GAE_LAMBDA": 0.95, "CLIP_EPS": 0.2, "ENTROPY_START_COEF": 0.10,
    "ENTROPY_ANNEAL_EPISODES": 20, "MAX_EPISODES_PER_PHASE": 500,
    "LOT_CURRICULUM_ENABLED": True, "START_PHASE": 0,
    "AUTO_TUNE_GPU": True, "USE_AMP": True, "USE_TORCH_COMPILE": True,
    "GPU_UTIL_TARGET": 0.80,
}


def _sanitize_label(label):
    """spaces->_, strip non [A-Za-z0-9_-], collapse repeats, 'unnamed' fallback."""
    import re
    s = (label or "").strip().replace(" ", "_")
    s = re.sub(r"[^A-Za-z0-9_\-]", "", s)
    s = re.sub(r"_+", "_", s).strip("_-")
    return s or "unnamed"


def _params_hash(params):
    blob = json.dumps(params, sort_keys=True, default=str)
    return hashlib.md5(blob.encode("utf-8")).hexdigest()[:8]


def _diff_from_defaults(params):
    """{key:{default,saved}} for every flat key that differs (REWARD unpacked)."""
    flat = dict(params)
    rew = flat.pop("REWARD", {}) or {}
    flat.update(rew)
    out = {}
    for k, dv in _DEFAULTS.items():
        if k in flat and flat[k] != dv:
            out[k] = {"default": dv, "saved": flat[k]}
    return out


def _read_log():
    """Read the master log array. On a CORRUPTED log, back it up to
    snapshot_log.backup.json and start fresh (warn), per the spec."""
    if not os.path.exists(_SNAPSHOT_LOG):
        return []
    try:
        with open(_SNAPSHOT_LOG) as f:
            log = json.load(f)
        if not isinstance(log, list):
            raise ValueError("snapshot_log.json is not a JSON array")
        return log
    except Exception as exc:
        backup = os.path.join(_SNAPSHOT_DIR, "snapshot_log.backup.json")
        try:
            os.replace(_SNAPSHOT_LOG, backup)
            print(f"⚠️  Corrupted snapshot_log.json ({exc}) — backed up to "
                  f"{backup}, starting fresh.")
        except Exception:
            print(f"⚠️  Corrupted snapshot_log.json ({exc}) — starting fresh.")
        return []


def _append_log(entry):
    """Append `entry` to the master snapshot_log.json array (create if missing).
    NEVER overwrites the whole file blindly: reads (corruption-safe), appends,
    writes back."""
    log = _read_log()
    log.append(entry)
    with open(_SNAPSHOT_LOG, "w") as f:
        json.dump(log, f, indent=2, default=str)
    return log


def _save_snapshot(label):
    """Write params_snapshot_<ts>_<label>.json + append to the master log."""
    try:
        PARAMS
    except NameError:
        print("⚠️  PARAMS not defined — run the dashboard cell (CELL 7) first.")
        return None
    os.makedirs(_SNAPSHOT_DIR, exist_ok=True)
    # Drive-not-mounted guard (the snapshot dir lives under the mounted Drive).
    if not os.path.isdir("/content/drive") and _SNAPSHOT_DIR.startswith("/content/drive"):
        display(HTML("<b style='color:#e53935'>❌ Google Drive not mounted. "
                     "Run: drive.mount('/content/drive')</b>"))
        return None
    now = datetime.now(timezone.utc)
    ts = now.strftime("%Y%m%d_%H%M%S")
    run_label = _sanitize_label(label)
    h = _params_hash(PARAMS)
    diff = _diff_from_defaults(PARAMS)
    changed_keys = sorted(diff.keys())
    # duplicate_of: the EARLIER entry's timestamp with the SAME hash (config same).
    dup = None
    for e in _read_log():
        if e.get("params_hash") == h:
            dup = e.get("timestamp")
            break
    meta = {
        "timestamp": now.isoformat(),
        "run_label": run_label,
        "colab_session": os.environ.get("COLAB_JUPYTER_IP", "unknown"),
        "params_hash": h,
    }
    blob = {
        "snapshot_meta": meta,
        "params": PARAMS,
        "diff_from_defaults": diff,
    }
    fname = f"params_snapshot_{ts}_{run_label}.json"
    fpath = os.path.join(_SNAPSHOT_DIR, fname)
    try:
        os.makedirs(_SNAPSHOT_DIR, exist_ok=True)
        with open(fpath, "w") as f:
            json.dump(blob, f, indent=2, default=str)
    except Exception as exc:                # write fail -> inline, never crash
        display(HTML(f"<b style='color:#e53935'>❌ Could not write snapshot: "
                     f"{exc}</b>"))
        return None
    entry = {"timestamp": meta["timestamp"], "run_label": run_label,
             "filename": fname, "params_hash": h,
             "n_changed_from_default": len(changed_keys),
             "changed_keys": changed_keys}
    if dup is not None:
        entry["duplicate_of"] = dup
    log = _append_log(entry)
    dup_note = (f"<br>🔁 duplicate_of <code>{dup}</code>" if dup else "")
    display(HTML(
        f"<div style='background:#1b5e20;color:#e8f5e9;padding:10px;"
        f"border-radius:8px;font-family:monospace'>✅ Saved snapshot<br>"
        f"file: <b>{fname}</b><br>path: {fpath}<br>"
        f"hash: <code>{h}</code> &nbsp; {len(changed_keys)} differing keys"
        f"<br>log now has {len(log)} entries{dup_note}</div>"))
    return fpath


def _view_log():
    log = _read_log()
    if not log:
        display(HTML("<i style='color:#888'>No snapshots saved yet.</i>"))
        return

    def _keys_cell(keys):
        keys = keys or []
        shown = ", ".join(keys[:3])
        extra = f" +{len(keys) - 3} more" if len(keys) > 3 else ""
        return shown + extra

    rows = []
    for i, e in enumerate(reversed(log)):           # newest-first
        n = len(log) - i
        is_dup = "duplicate_of" in e
        bg = "background:#3a3000;" if is_dup else ""
        badge = " 🔁" if is_dup else ""
        rows.append(
            f"<tr style='{bg}'><td style='padding:2px 8px'>{n}</td>"
            f"<td style='padding:2px 8px'>{e.get('timestamp','')}{badge}</td>"
            f"<td style='padding:2px 8px'>{e.get('run_label','')}</td>"
            f"<td style='padding:2px 8px'><code>{e.get('params_hash','')}</code></td>"
            f"<td style='padding:2px 8px'>{e.get('n_changed_from_default',0)}</td>"
            f"<td style='padding:2px 8px'>{_keys_cell(e.get('changed_keys'))}</td></tr>")
    display(HTML(
        "<div style='background:#272735;color:#e6e6f0;padding:10px;"
        "border-radius:8px;font-family:monospace;font-size:12px'>"
        f"<b>Snapshot Log — {len(log)} entries (newest first)</b>"
        "<table style='border-collapse:collapse;margin-top:6px'>"
        "<tr><th style='text-align:left;padding:2px 8px'>#</th>"
        "<th style='text-align:left;padding:2px 8px'>Timestamp</th>"
        "<th style='text-align:left;padding:2px 8px'>Run Label</th>"
        "<th style='text-align:left;padding:2px 8px'>Hash</th>"
        "<th style='text-align:left;padding:2px 8px'>Changed</th>"
        "<th style='text-align:left;padding:2px 8px'>Keys Modified</th></tr>"
        + "".join(rows) + "</table></div>"))


_label_box = W.Text(placeholder="e.g. aggressive_target_3pct",
                    description="Run label:",
                    style={"description_width": "initial"},
                    layout=W.Layout(width="55%"))
_path_box = W.Text(value=_SNAPSHOT_DIR, description="Save to:",
                   style={"description_width": "initial"},
                   layout=W.Layout(width="80%"))
_save_btn = W.Button(description="💾 Save Snapshot", button_style="success")
_log_btn = W.Button(description="📋 View Snapshot Log", button_style="")
_save_out = W.Output()


def _on_save(_):
    # Honor an edited Save-to path (recompute the log path with it).
    global _SNAPSHOT_DIR, _SNAPSHOT_LOG
    _SNAPSHOT_DIR = (_path_box.value or _SNAPSHOT_DIR).strip()
    _SNAPSHOT_LOG = os.path.join(_SNAPSHOT_DIR, "snapshot_log.json")
    with _save_out:
        _save_out.clear_output()
        _save_snapshot(_label_box.value)


def _on_view(_):
    global _SNAPSHOT_DIR, _SNAPSHOT_LOG
    _SNAPSHOT_DIR = (_path_box.value or _SNAPSHOT_DIR).strip()
    _SNAPSHOT_LOG = os.path.join(_SNAPSHOT_DIR, "snapshot_log.json")
    with _save_out:
        _save_out.clear_output()
        _view_log()


_save_btn.on_click(_on_save)
_log_btn.on_click(_on_view)

# Green/teal bordered panel (Task C).
display(HTML("<div style='background:#00695c;color:#e0f2f1;padding:8px 12px;"
             "border-radius:6px 6px 0 0;font-weight:700;"
             "border:2px solid #26a69a'>💾 Save Parameter Snapshot</div>"))
_panel = W.VBox([_label_box, _path_box, W.HBox([_save_btn, _log_btn]), _save_out],
                layout=W.Layout(border="2px solid #26a69a", padding="8px"))
display(_panel)


In [ ]:
# CELL 8 — LAUNCH JORDAN DASHBOARD (localtunnel; ngrok fallback)
import subprocess, sys, time
CLONE_DIR = '/content/rl-trading-live'
subprocess.Popen([sys.executable, '-m', 'streamlit', 'run', 'dashboard/app.py',
    '--server.port', '8501', '--server.headless', 'true'], cwd=CLONE_DIR)
time.sleep(6)
# localtunnel is flaky on Colab; if the URL doesn't load, use the ngrok block below.
subprocess.run(['npx', 'localtunnel', '--port', '8501'], check=False)
# Fallback (uncomment): 
# !pip -q install pyngrok && python -c "from pyngrok import ngrok; print(ngrok.connect(8501))"

In [ ]:
# CELL 9 — CRASH RECOVERY (run if training crashed)
import subprocess, sys
CLONE_DIR = '/content/rl-trading-live'
subprocess.run([sys.executable, 'scripts/crash_recovery.py',
    '--checkpoint-dir', '/content/drive/MyDrive/RL-Trading-Checkpoints/gpu',
    '--manifest', '/content/drive/MyDrive/RL-Trading-Checkpoints/manifest.json',
], cwd=CLONE_DIR, check=True)
# Then re-run CELL 7.

In [ ]:
# CELL 10 — GPU PROFILING (run after CELL 7 has trained a few episodes)
import torch, sys, os
sys.path.insert(0, '/content/rl-trading-live')
os.chdir('/content/rl-trading-live')
from torch.profiler import profile, record_function, ProfilerActivity
from core.pipeline import build_pipeline
from core.settings import CFG, get_device, auto_tune_batch

device = get_device()
cfg = auto_tune_batch(dict(CFG), device)
cfg['DATA_CSV_EURUSD'] = '/content/drive/MyDrive/RL-Trading-Data/EURUSD_M1_202101131130_202605270000_2020_2026.csv'
env, agent, *_ = build_pipeline(cfg, device,
    phase={'name': 'profile', 'entry_conditions': {'buy': 'any', 'sell': 'any'}})

obs = torch.randn(cfg['BATCH_SIZE_ENV'], env.state_dim, device=device)
with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
             record_shapes=True) as prof:
    with record_function('actor_critic_forward'):
        with torch.amp.autocast('cuda', enabled=device.type == 'cuda'):
            out = agent.net(obs)   # agent.net is the ActorCritic module

print(prof.key_averages().table(sort_by='cuda_time_total', row_limit=15))
cuda_t = sum(e.cuda_time for e in prof.key_averages())
cpu_t  = sum(e.cpu_time  for e in prof.key_averages())
ratio  = cuda_t / (cuda_t + cpu_t + 1e-9)
print(f'\nGPU time ratio: {ratio:.1%}')
print('GPU utilization OK' if ratio >= 0.5 else
      'WARNING: GPU < 50% — raise BATCH_SIZE_ENV in core/settings.py')

In [ ]:
# CELL 11 — 🔍 POLICY INTERPRETABILITY (post-hoc; ZERO training overhead)
# ─────────────────────────────────────────────────────────────────────────────
# RUN-ALL SAFE + graceful skip. Loads the best checkpoint, builds a representative
# observation batch from a short env rollout, and ALWAYS runs the FAST path
# (no SHAP needed):
#     1) SALIENCY      top features per head (torch.autograd.grad, <5s/10k obs)
#     2) ACTION DIST   the policy's BUY/SELL/FLAT + exit + lot mix
#     3) POLICY REPORT plain-English .txt + .json (personality/session/risk/…)
# Then, ONLY if RUN_SHAP is True AND `shap` is installed, it runs the (slower)
# SHAP explainer (cached by checkpoint hash). SHAP is NEVER imported by training.
#
# If the checkpoint or data is missing, the cell prints a one-liner and SKIPS —
# it never breaks a Run-All.
import os

RUN_SHAP = False        # ← toggle on (and `pip install shap`) for the SHAP section

DRIVE        = "/content/drive/MyDrive"
CKPT_DIR     = f"{DRIVE}/RL-Trading-Checkpoints/gpu"
METRICS_DIR  = f"{DRIVE}/RL-Trading-Checkpoints/metrics"
CSV_PATH     = f"{DRIVE}/RL-Trading-Data/EURUSD_M1_202101131130_202605270000_2020_2026.csv"
CKPT_PATH    = f"{CKPT_DIR}/best_eval.pt"
N_OBS        = 2000     # observations to analyze (sampled from a fresh rollout)


def _interpret():
    import torch
    if not os.path.exists(CKPT_PATH):
        print(f"ℹ️  No checkpoint at {CKPT_PATH} — train first, then re-run. Skipping.")
        return
    try:
        from core.settings import CFG
        from core.pipeline import build_pipeline
        from core.interpret.saliency import saliency_from_checkpoint, save_saliency_bars
        from core.interpret.action_logger import action_distribution
        from core.interpret.policy_report import generate_policy_report
        from core.interpret.dashboard_utils import obs_feature_names
        from core.env.indicators import FEATURE_COLUMNS
    except Exception as exc:
        print(f"ℹ️  interpret deps unavailable ({exc}) — skipping.")
        return

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    cfg = dict(CFG)
    cfg["DATA_CSV_EURUSD"] = CSV_PATH

    # Build a representative observation batch from a short rollout of the env.
    try:
        env, agent, _sizer, _guard, _gate = build_pipeline(cfg, device, phase=None)
        agent.load(CKPT_PATH, partial=True)
        state = env.reset()
        obs_list = [state.detach().cpu()]
        steps = max(1, N_OBS // max(1, state.shape[0]))
        with torch.no_grad():
            for _ in range(steps):
                out = agent.select_actions(state, mask=env.current_direction_mask())
                state, _r, _d, _info = env.step(out)
                obs_list.append(state.detach().cpu())
        obs = torch.cat(obs_list, dim=0)[:N_OBS]
    except Exception as exc:
        print(f"ℹ️  could not build obs batch ({exc}) — skipping.")
        return

    lkbk = int(cfg.get("LOOKBACK", 20))
    n_ind = max(1, (obs.shape[-1] - 20) // max(lkbk, 1))
    icols = list(FEATURE_COLUMNS)

    print("═" * 70)
    print("  🔍 FAST PATH (always runs — no SHAP)")
    print("═" * 70)

    # 1) SALIENCY ────────────────────────────────────────────────────────────
    sal = saliency_from_checkpoint(CKPT_PATH, obs, cfg, device=device,
                                   indicator_columns=icols)
    for head, res in sal.items():
        top = ", ".join(f"{fn}({v:.3f})" for fn, v in res["ranking"][:5])
        print(f"  saliency[{head:9}] top5: {top}")
    try:
        bars = save_saliency_bars(sal, METRICS_DIR)
        print(f"  saliency bars -> {bars}")
    except Exception:
        pass

    # 2) ACTION DISTRIBUTION ───────────────────────────────────────────────────
    with torch.no_grad():
        dl, el, lm, _v = agent._fwd(obs.to(device).float())
        dist = action_distribution(dl, el, torch.sigmoid(lm.squeeze(-1)))
    print(f"  action mix: BUY {dist['dir_BUY']*100:.1f} / "
          f"SELL {dist['dir_SELL']*100:.1f} / FLAT {dist['dir_FLAT']*100:.1f}  | "
          f"lot μ={dist['lot_mean']:.3f} σ={dist['lot_std']:.3f}")

    # 3) POLICY REPORT ─────────────────────────────────────────────────────────
    rep = generate_policy_report(CKPT_PATH, obs, cfg, METRICS_DIR,
                                 indicator_columns=icols, device=device)
    print("\n" + rep["text"])
    print(f"\n  report files -> {rep['paths']}")

    # ── OPTIONAL SHAP (post-hoc, cached; skipped unless RUN_SHAP and installed) ─
    if not RUN_SHAP:
        print("\nℹ️  RUN_SHAP is False — skipping SHAP (fast path complete).")
        return
    from core.interpret.shap_explain import shap_available, run_shap
    if not shap_available():
        print("\nℹ️  RUN_SHAP is True but `shap` is not installed "
              "(`pip install shap`) — skipping SHAP.")
        return
    print("\n" + "═" * 70)
    print("  🧬 SHAP (GradientExplainer, cached by checkpoint hash)")
    print("═" * 70)
    bg_n = int(cfg.get("SHAP_BACKGROUND_SAMPLES", 256))
    ex_n = int(cfg.get("SHAP_EXPLAIN_SAMPLES", 200))
    background = obs[:bg_n]
    explain = obs[bg_n:bg_n + ex_n] if obs.shape[0] > bg_n else obs[:ex_n]
    result = run_shap(CKPT_PATH, background, explain, cfg, METRICS_DIR,
                      device=device, indicator_columns=icols)
    for head, res in result.items():
        imp = res["importances"]
        fnames = res["feature_names"]
        order = imp.argsort()[::-1][:5]
        top = ", ".join(f"{fnames[i]}({imp[i]:.3f})" for i in order)
        cached = " (cached)" if res.get("cached") else ""
        print(f"  shap[{head:9}] top5{cached}: {top}")


_interpret()
